# MIR — Reranking SIFT sur tous les modèles

Ce notebook applique un **reranking par SIFT** aux embeddings produits par tous les modèles testés :
1. DINOv2 (zero-shot + SupCon finetuned)
2. MobileNetV2 (zero-shot + SupCon finetuned)
3. EfficientNet-B0 (zero-shot + SupCon finetuned)
4. EfficientNet-B4 (zero-shot + SupCon finetuned)
5. ResNet-50 (zero-shot)

## Méthodologie

Pour chaque image query :
1. **Retrieval global** : on récupère le top-K candidats via similarité cosine sur les embeddings (rapide)
2. **Reranking SIFT** : on calcule SIFT sur la query + les K candidats, on compte les matches géométriquement cohérents (RANSAC), on réordonne selon ce score (précis)

Calcul SIFT **à la volée** sur les K candidats seulement, pas d'index pré-calculé.

In [ ]:
import os
import cv2
import numpy as np
from tqdm import tqdm

# Le dossier contenant toutes vos images (ou un dossier parent avec des sous-dossiers)
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
IMAGES_DIR = os.path.join(PROJECT_ROOT, "dataset")

# Où sauvegarder les nouveaux index SIFT
OUTPUT_NPZ_PATH = os.path.join(PROJECT_ROOT, "indexes", "sift_local_features.npz")
OUTPUT_FILENAMES_PATH = os.path.join(PROJECT_ROOT, "indexes", "sift_local_filenames.npy")

# Paramètres SIFT (ajustables selon vos besoins)
MAX_KEYPOINTS = 500  # Limiter le nombre de points pour économiser du stockage/temps de calcul
# ==============================================================================

def extract_sift_features(images_dir, max_keypoints=500):
    """
    Extrait les descripteurs SIFT locaux pour toutes les images d'un dossier.
    Gère les sous-dossiers.
    """
    print(f"Début de l'extraction SIFT depuis : {images_dir}")

    # 1. Lister toutes les images (gestion des sous-dossiers)
    all_image_paths = []
    for root, _, files in os.walk(images_dir):
        for file in files:
            if file.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp')):
                all_image_paths.append(os.path.join(root, file))


    all_image_paths.sort()

    if not all_image_paths:
        print("Erreur : Aucune image trouvée dans le répertoire.")
        return None, None

    print(f"Nombre total d'images trouvées : {len(all_image_paths)}")

    # Initialiser l'extracteur SIFT
    # Limiter le nombre de points clés (nfeatures) est crucial pour ne pas exploser la RAM et le stockage
    sift = cv2.SIFT_create(nfeatures=max_keypoints)

    sift_dict = {}
    valid_filenames = []

    # 2. Boucle d'extraction avec barre de progression
    for img_path in tqdm(all_image_paths, desc="Extraction SIFT locale"):
        # Lire l'image en niveaux de gris (SIFT travaille sur la luminance)
        img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)

        # Pour une raison quelconque, l'image n'a pas pu être lue
        if img is None:
            print(f"\nAttention : Impossible de lire l'image {img_path}. Ignorée.")
            continue

        # Extraire les points clés et les descripteurs
        keypoints, descriptors = sift.detectAndCompute(img, None)

        filename = os.path.basename(img_path)

        # Si aucun point clé n'est trouvé (image trop uniforme)
        if descriptors is None or len(keypoints) == 0:
            # On stocke des tableaux vides pour garder la correspondance
            sift_dict[f"{filename}_kp"] = np.empty((0, 2), dtype=np.float32)
            sift_dict[f"{filename}_des"] = np.empty((0, 128), dtype=np.float32)
        else:
            # RANSAC a besoin des coordonnées (x,y) des points clés.
            # On extrait uniquement la propriété 'pt' des objets cv2.KeyPoint
            kp_coords = np.array([kp.pt for kp in keypoints], dtype=np.float32)

            sift_dict[f"{filename}_kp"] = kp_coords
            sift_dict[f"{filename}_des"] = descriptors

        valid_filenames.append(filename)

    return sift_dict, valid_filenames

# Exécution
print("--- Démarrage du script ---")
features_dict, filenames_list = extract_sift_features(IMAGES_DIR, MAX_KEYPOINTS)

if features_dict and filenames_list:
    # Sauvegarde des données
    print(f"Sauvegarde des caractéristiques SIFT dans : {OUTPUT_NPZ_PATH}")
    # **kwargs pour décompresser le dictionnaire en arguments nommés
    np.savez_compressed(OUTPUT_NPZ_PATH, **features_dict)

    print(f"Sauvegarde de la liste des fichiers dans : {OUTPUT_FILENAMES_PATH}")
    np.save(OUTPUT_FILENAMES_PATH, np.array(filenames_list))

    print("--- Extraction terminée avec succès ! ---")

    # Petit test de vérification
    test_file = filenames_list[0]
    print(f"\nVérification sur le premier fichier ({test_file}) :")
    print(f"  Shape Keypoints : {features_dict[f'{test_file}_kp'].shape}")
    print(f"  Shape Descriptors : {features_dict[f'{test_file}_des'].shape}")

In [ ]:
!pip install opencv-contrib-python -q

In [ ]:

import os

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
IMAGE_DIR     = os.path.join(PROJECT_ROOT, "dataset")
RESULTS_DIR   = os.path.join(PROJECT_ROOT, "results", "reranking_sift")
os.makedirs(RESULTS_DIR, exist_ok=True)

# Chemins vers les fichiers .pth d'embeddings de chaque modèle
# Si tu n'as pas un modèle, mets None pour le sauter
EMBEDDING_PATHS = {
    # DINOv2
    'DINOv2_ZeroShot':        os.path.join(PROJECT_ROOT, 'DinoV2_training', 'DinoV2_best.pth'),
    'DINOv2_SupCon':          os.path.join(PROJECT_ROOT, 'DinoV2_training', 'DinoV2_finetuned_best.pth'),

    # MobileNetV2
    'MobileNetV2_ZeroShot':   os.path.join(PROJECT_ROOT, 'MobileNetV2_training', 'MobileNetV2_zero_shot.pth'),
    'MobileNetV2_SupCon':     os.path.join(PROJECT_ROOT, 'MobileNetV2_training', 'MobileNetV2_finetuned_best.pth'),


    # ResNet-50
    'ResNet50_ZeroShot':       os.path.join(PROJECT_ROOT, 'ResNet50_training', 'ResNet50_zero_shot.pth'),

    # ViT-Base/16
    'ViTBase16_ZeroShot':      os.path.join(PROJECT_ROOT, 'VitB16_training', 'ViT_B_16_zero_shot.pth'),
}

# PARAMÈTRES DE RERANKING
TOP_K_RETRIEVE = 50    # nb candidats récupérés par le retrieval global
K_values       = [1, 5, 20, 50]

# Filtrer les chemins existants
EMBEDDING_PATHS = {k: v for k, v in EMBEDDING_PATHS.items()
                   if v is not None and os.path.exists(v)}

print(f'Dataset : {IMAGE_DIR}')
print(f'Résultats : {RESULTS_DIR}')
print(f'\n{len(EMBEDDING_PATHS)} modèles trouvés :')
for name, path in EMBEDDING_PATHS.items():
    print(f'  ✓ {name}')

# Vérifier les modèles manquants
all_expected = [
    'DINOv2_ZeroShot', 'DINOv2_SupCon',
    'MobileNetV2_ZeroShot', 'MobileNetV2_SupCon',
    'EfficientNetB0_ZeroShot', 'EfficientNetB0_SupCon',
    'EfficientNetB4_ZeroShot', 'EfficientNetB4_SupCon',
    'ResNet50_ZeroShot', 'ViTBase16_ZeroShot',
]
missing = [m for m in all_expected if m not in EMBEDDING_PATHS]
if missing:
    print(f'\nModèles manquants (chemins introuvables) :')
    for m in missing:
        print(f'  ✗ {m}')

In [ ]:
import torch
import torch.nn.functional as F
import numpy as np
import cv2
from PIL import Image
from tqdm import tqdm
import matplotlib.pyplot as plt
import json

# Vérifier que SIFT est dispo
_ = cv2.SIFT_create()
print('SIFT disponible ✓')

def get_label_from_filename(filename):
    name_without_ext = filename.split('.')[0]
    parts = name_without_ext.split('_')
    if len(parts) >= 4:
        return f'{parts[2]}_{parts[3]}'
    return 'Label_Inconnu'

## 1. Fonction de reranking SIFT

Pour chaque paire (query, candidat) :
1. Extraction des keypoints + descripteurs SIFT
2. Matching par BFMatcher avec ratio test de Lowe (filtre les matches ambigus)
3. Vérification géométrique par RANSAC (homographie) pour ne garder que les inliers
4. Score = nombre d'inliers

In [ ]:

# Un seul détecteur SIFT, réutilisé
sift = cv2.SIFT_create(nfeatures=500)  # limite à 500 keypoints par image (rapidité)
bf   = cv2.BFMatcher(cv2.NORM_L2)

# Cache des descripteurs SIFT pour ne pas recalculer la même image plusieurs fois
SIFT_CACHE = {}

def load_gray_image(filename):
    """Charge une image en niveaux de gris pour SIFT."""
    path = os.path.join(IMAGE_DIR, filename)
    img  = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
    return img

def get_sift_features(filename):
    """Récupère (keypoints, descriptors) avec cache."""
    if filename in SIFT_CACHE:
        return SIFT_CACHE[filename]
    img = load_gray_image(filename)
    if img is None:
        SIFT_CACHE[filename] = (None, None)
        return None, None
    kp, des = sift.detectAndCompute(img, None)
    SIFT_CACHE[filename] = (kp, des)
    return kp, des

def sift_match_score(query_filename, candidate_filename,
                    ratio=0.75, ransac_threshold=5.0):
    """Retourne le nombre d'inliers SIFT entre query et candidat."""
    kp1, des1 = get_sift_features(query_filename)
    kp2, des2 = get_sift_features(candidate_filename)

    if des1 is None or des2 is None or len(des1) < 4 or len(des2) < 4:
        return 0

    # Matching avec ratio test de Lowe
    try:
        raw_matches = bf.knnMatch(des1, des2, k=2)
    except cv2.error:
        return 0

    good = []
    for pair in raw_matches:
        if len(pair) < 2:
            continue
        m, n = pair
        if m.distance < ratio * n.distance:
            good.append(m)

    if len(good) < 4:
        return len(good)

    # Vérification géométrique RANSAC
    src_pts = np.float32([kp1[m.queryIdx].pt for m in good]).reshape(-1, 1, 2)
    dst_pts = np.float32([kp2[m.trainIdx].pt for m in good]).reshape(-1, 1, 2)

    try:
        _, mask = cv2.findHomography(src_pts, dst_pts,
                                     cv2.RANSAC, ransac_threshold)
        inliers = int(mask.sum()) if mask is not None else 0
    except cv2.error:
        inliers = 0

    return inliers

print('Fonctions SIFT prêtes')

## 2. Évaluation avec et sans reranking

In [ ]:

def evaluate_recall_and_map(retrieved_per_query, labels, K_values):
    """
    retrieved_per_query : liste de listes d'indices (top-K déjà ordonnés par modèle)
    labels              : liste de labels pour toutes les images
    Retourne (recall_scores_dict, mAP)
    """
    recalls   = {k: 0 for k in K_values}
    ap_scores = []
    n         = len(retrieved_per_query)

    label_counts = {}
    for lbl in labels:
        label_counts[lbl] = label_counts.get(lbl, 0) + 1

    for i, top_indices in enumerate(retrieved_per_query):
        query_label      = labels[i]
        retrieved_labels = [labels[idx] for idx in top_indices]


        for k in K_values:
            if query_label in retrieved_labels[:k]:
                recalls[k] += 1


        num_relevant = label_counts[query_label] - 1
        if num_relevant == 0:
            continue

        hits, precision_sum = 0, 0.0
        for rank, lbl in enumerate(retrieved_labels, start=1):
            if lbl == query_label:
                hits += 1
                precision_sum += hits / rank
        ap_scores.append(precision_sum / num_relevant)

    recall_scores = {k: recalls[k] / n * 100 for k in K_values}
    mAP           = sum(ap_scores) / len(ap_scores) * 100 if ap_scores else 0.0
    return recall_scores, mAP

In [ ]:
import os
import json
import torch
import numpy as np
import cv2
from tqdm import tqdm

SIFT_NPZ_PATH = os.path.join(PROJECT_ROOT, "indexes", "sift_local_features.npz")
RESULTS_PATH  = os.path.join(PROJECT_ROOT, "results", "reranking_results.json")

# --- HYPERPARAMÈTRE : Le bassin de candidats ---
# +25% de candidats (Ex: si TOP_K = 50, on donne 62 images à SIFT pour le repêchage)
POOL_SIZE = TOP_K_RETRIEVE + (TOP_K_RETRIEVE // 4)

print(f"Chargement des descripteurs SIFT locaux depuis : {SIFT_NPZ_PATH}")
sift_data = np.load(SIFT_NPZ_PATH)

def compute_sift_ransac_inliers(kp1, des1, kp2, des2):
    """Calcule le nombre d'inliers via RANSAC après matching SIFT."""
    if des1 is None or des2 is None or len(des1) < 4 or len(des2) < 4:
        return 0

    bf = cv2.BFMatcher(cv2.NORM_L2, crossCheck=False)
    try:
        matches = bf.knnMatch(des1, des2, k=2)
    except Exception:
        return 0

    # Ratio test de Lowe
    good_matches = [m for match_pair in matches if len(match_pair) == 2
                    for m, n in [match_pair] if m.distance < 0.75 * n.distance]

    if len(good_matches) >= 4:
        src_pts = np.float32([kp1[m.queryIdx] for m in good_matches]).reshape(-1, 1, 2)
        dst_pts = np.float32([kp2[m.trainIdx] for m in good_matches]).reshape(-1, 1, 2)
        try:
            _, mask = cv2.findHomography(src_pts, dst_pts, cv2.RANSAC, 5.0)
            if mask is not None:
                return int(np.sum(mask))
        except Exception:
            return 0
    return 0

# 1. CHARGEMENT DE L'HISTORIQUE (Optionnel, sécurise vos données)
all_results = {}
if os.path.exists(RESULTS_PATH):
    print(f"\n📥 Chargement du fichier existant : {os.path.basename(RESULTS_PATH)}...")
    with open(RESULTS_PATH, 'r') as f:
        all_results = json.load(f)

# ── Évaluation ────────────────────────────────────────────────────────────────

# La boucle parcourt TOUS vos modèles
for model_name, emb_path in EMBEDDING_PATHS.items():
    print(f'\n{"="*60}')
    print(f'Modèle : {model_name} (Pool RANSAC : {POOL_SIZE} candidats)')
    print(f'{"="*60}')

    db         = torch.load(emb_path)
    embeddings = db['embeddings']
    filenames  = [os.path.basename(str(f)) for f in db['filenames']]
    labels     = [get_label_from_filename(f) for f in filenames]
    n          = len(embeddings)

    # 1. Retrieval global (On extrait POOL_SIZE, soit 25% de plus)
    sim_matrix = torch.mm(embeddings, embeddings.T)
    sim_matrix.fill_diagonal_(-1.0)
    _, top_indices_global = torch.topk(sim_matrix, POOL_SIZE, dim=1)
    top_indices_global    = top_indices_global.tolist()

    # Évaluation de la baseline sans reranking (bridée strictement au TOP_K d'origine)
    baseline_indices = [cands[:TOP_K_RETRIEVE] for cands in top_indices_global]
    recall_no_rerank, map_no_rerank = evaluate_recall_and_map(baseline_indices, labels, K_values)

    print(f'\n[Sans reranking]')
    for k, v in recall_no_rerank.items():
        print(f'  Recall@{k:>2} : {v:.2f}%')
    print(f'  mAP        : {map_no_rerank:.2f}%')

    # 2. Reranking SIFT + RANSAC sur le pool élargi
    print(f'\n[Reranking SIFT + RANSAC sur {POOL_SIZE} candidats]')
    top_indices_reranked = []

    for i in tqdm(range(n), desc=f'Reranking {model_name}'):
        candidates = top_indices_global[i]
        query_file = filenames[i]

        try:
            query_kp  = sift_data[f"{query_file}_kp"]
            query_des = sift_data[f"{query_file}_des"]
        except KeyError:
            query_kp, query_des = None, None

        sift_scores = []
        for c_idx in candidates:
            cand_file = filenames[c_idx]
            try:
                cand_kp  = sift_data[f"{cand_file}_kp"]
                cand_des = sift_data[f"{cand_file}_des"]
            except KeyError:
                cand_kp, cand_des = None, None

            score = compute_sift_ransac_inliers(query_kp, query_des, cand_kp, cand_des)
            sift_scores.append(score)

        # Tri décroissant : les candidats avec le plus d'inliers passent en premier
        order = sorted(range(len(candidates)), key=lambda x: sift_scores[x], reverse=True)
        reranked = [candidates[o] for o in order]
        top_indices_reranked.append(reranked)

    # 3. Évaluation AVEC reranking (la fonction evaluate_recall_and_map coupera d'elle-même à K)
    recall_rerank, map_rerank = evaluate_recall_and_map(top_indices_reranked, labels, K_values)

    print(f'\n[Avec reranking SIFT + RANSAC]')
    for k, v in recall_rerank.items():
        delta = v - recall_no_rerank[k]
        print(f'  Recall@{k:>2} : {v:.2f}%  ({"+" if delta >= 0 else ""}{delta:.2f})')
    print(f'  mAP        : {map_rerank:.2f}%  ({"+" if map_rerank-map_no_rerank>=0 else ""}{map_rerank-map_no_rerank:.2f})')

    # 4. Ajout ou écrasement au dictionnaire global
    all_results[model_name] = {
        'no_rerank': {'recall': recall_no_rerank, 'mAP': map_no_rerank},
        'rerank_sift': {'recall': recall_rerank, 'mAP': map_rerank},
    }

# 5. SAUVEGARDE GLOBALE
with open(RESULTS_PATH, 'w') as f:
    json.dump(all_results, f, indent=2, default=str)
print(f'\n✅ Évaluation totale terminée. Fichier mis à jour avec succès : {RESULTS_PATH}')

## 3. Visualisation des résultats

In [ ]:
import os
import json



results_path = os.path.join(RESULTS_DIR, 'reranking_results.json')

if not os.path.exists(results_path):
    print(f"❌ Impossible de trouver le fichier de résultats : {results_path}")
else:
    print(f"📥 Chargement des résultats depuis : {results_path}\n")

    # Charger les données depuis le Drive
    with open(results_path, 'r') as f:
        drive_results = json.load(f)

    print(f'{"Modèle":<28} {"R@1":>10} {"R@5":>10} {"R@20":>10} {"R@50":>10} {"mAP":>10}')
    print('-' * 80)

    for model_name, res in drive_results.items():
        nr = res['no_rerank']
        rr = res['rerank_sift']

        # Utilisation de .get("X", 0) pour éviter les erreurs si une valeur manque,
        # et utilisation de chaînes de caractères ("1") car le format JSON transforme les entiers en strings.
        print(f'{model_name:<28} '
              f'{nr["recall"].get("1", 0):>9.1f}% '
              f'{nr["recall"].get("5", 0):>9.1f}% '
              f'{nr["recall"].get("20", 0):>9.1f}% '
              f'{nr["recall"].get("50", 0):>9.1f}% '
              f'{nr["mAP"]:>9.2f}%')

        print(f'{"  + SIFT rerank":<28} '
              f'{rr["recall"].get("1", 0):>9.1f}% '
              f'{rr["recall"].get("5", 0):>9.1f}% '
              f'{rr["recall"].get("20", 0):>9.1f}% '
              f'{rr["recall"].get("50", 0):>9.1f}% '
              f'{rr["mAP"]:>9.2f}%')
        print()

In [ ]:
import os
import json
import numpy as np
import matplotlib.pyplot as plt


results_path = os.path.join(RESULTS_DIR, 'reranking_results.json')

if not os.path.exists(results_path):
    print(f"❌ Impossible de trouver le fichier de résultats : {results_path}")
else:
    print(f"📥 Chargement des résultats depuis : {results_path}")

    # Charger les données depuis le Drive
    with open(results_path, 'r') as f:
        drive_results = json.load(f)

    model_names    = list(drive_results.keys())

    # ⚠️ Attention : dans le JSON, les clés numériques (comme 1) deviennent des strings ("1")
    recall1_no     = [drive_results[m]['no_rerank']['recall'].get('1', 0)   for m in model_names]
    recall1_rerank = [drive_results[m]['rerank_sift']['recall'].get('1', 0) for m in model_names]

    x     = np.arange(len(model_names))
    width = 0.35

    fig, ax = plt.subplots(figsize=(14, 6))
    bars1 = ax.bar(x - width/2, recall1_no,     width, label='Sans reranking',   color='#1f77b4')
    bars2 = ax.bar(x + width/2, recall1_rerank, width, label='Avec SIFT rerank', color='#ff7f0e')

    for bars in (bars1, bars2):
        for b in bars:
            h = b.get_height()
            ax.annotate(f'{h:.1f}', xy=(b.get_x() + b.get_width()/2, h),
                        xytext=(0, 3), textcoords='offset points',
                        ha='center', fontsize=8)

    ax.set_ylabel('Recall@1 (%)', fontsize=12)
    ax.set_title('Impact du reranking SIFT sur Recall@1 — Tous modèles',
                 fontsize=14, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels(model_names, rotation=30, ha='right')
    ax.set_ylim(0, 105)
    ax.grid(True, linestyle='--', alpha=0.5, axis='y')
    ax.legend()
    plt.tight_layout()

    save_path = os.path.join(RESULTS_DIR, 'impact_SIFT_recall1.png')
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    print(f'✅ Graphe sauvegardé : {save_path}')
    plt.show()

In [ ]:
import os
import json
import matplotlib.pyplot as plt


# Chemins vers vos deux fichiers distincts sur le Drive
path_principal  = os.path.join(RESULTS_DIR, 'reranking_results.json')

if not os.path.exists(results_path):
    print(f"❌ Impossible de trouver le fichier de résultats : {results_path}")
else:
    print(f"📥 Chargement des résultats depuis : {results_path}")

    with open(path_principal, 'r') as f:
      drive_results = json.load(f)


    # Récupérer les clés K (les valeurs de l'axe X) depuis le premier modèle trouvé
    # On les trie en entiers pour que l'axe X soit dans le bon ordre (1, 5, 20, 50)
    first_model_data = list(drive_results.values())[0]
    string_k_values = list(first_model_data['no_rerank']['recall'].keys())
    K_values_sorted = sorted([int(k) for k in string_k_values])

    # 2. Préparation de la figure
    n_models = len(drive_results)
    n_cols   = 2
    n_rows   = (n_models + n_cols - 1) // n_cols

    fig, axes = plt.subplots(n_rows, n_cols, figsize=(14, 4 * n_rows))
    axes = axes.flatten() if n_models > 1 else [axes]

    # 3. Traçage des courbes pour chaque modèle
    for idx, (model_name, res) in enumerate(drive_results.items()):
        ax = axes[idx]

        # Extraction sécurisée des valeurs en repassant l'entier k en string ("1", "5"...)
        no_vals  = [res['no_rerank']['recall'].get(str(k), 0)   for k in K_values_sorted]
        rr_vals  = [res['rerank_sift']['recall'].get(str(k), 0) for k in K_values_sorted]

        ax.plot(K_values_sorted, no_vals, marker='o', linewidth=2,
                label=f'Baseline (mAP {res["no_rerank"]["mAP"]:.1f}%)')
        ax.plot(K_values_sorted, rr_vals, marker='s', linewidth=2,
                label=f'+ SIFT (mAP {res["rerank_sift"]["mAP"]:.1f}%)')

        ax.set_title(model_name, fontsize=12, fontweight='bold')
        ax.set_xlabel('K')
        ax.set_ylabel('Recall (%)')
        ax.set_xticks(K_values_sorted)
        ax.set_ylim(0, 105)
        ax.grid(True, linestyle='--', alpha=0.5)
        ax.legend(fontsize=9)

    # Cacher les axes vides (si le nombre de modèles est impair)
    for j in range(len(drive_results), len(axes)):
        axes[j].set_visible(False)

    plt.tight_layout()

    # Sauvegarde sur le Drive
    save_path = os.path.join(RESULTS_DIR, 'all_models_recall_comparison.png')
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    print(f'✅ Graphe sauvegardé : {save_path}')

    plt.show()